# Imputation

Handling **missing values** to preserve data integrity and enable model training.

**Data cleaning** is the broad process of fixing or removing incorrect, corrupted, or incomplete data

vs 

**Imputation** is a *specific* cleaning technique that fills in missing values with estimated ones.

### Key Methods:
- Statistical substitution (mean, median, mode)
- Proximity-based methods (KNN, interpolation)
- Model-based prediction

In [10]:
# Import necessary example packages
import numpy as np
import pandas as pd

## Statistical substitution 

Simple replacement using column statistics.

Fill NAs with:
>- **mean**
>- **median**
>- **mode**

Scikit-Learn provides the `Imputer` class:

In [11]:
# Import SimpleImputer from Imputer class
from sklearn.impute import SimpleImputer

#### Example with Mean:

In [12]:
# create sample data with missing values
df = pd.DataFrame(np.random.randint(0,100, (5, 5)), columns=list('ABCDE'))

# randomly set some values to NaN
for index, row in df.iterrows():
    for col in df.columns:
        # randomly set ~30% of values to NaN
        if np.random.random() < 0.3:
            df.loc[index, col] = np.nan
            
# save copy for Imputer class example
df_copy = df.copy()

df

,A,B,C,D,E
0,62.0,16,NaN,NaN,57.0
1,84.0,8,NaN,49.0,NaN
2,NaN,48,NaN,5.0,28.0
3,NaN,64,31.0,43.0,81.0
4,36.0,26,81.0,NaN,54.0


In [13]:
# Imputation with mean (using Pandas)
for col in df.columns:
    column_mean = df[col].mean()
    df[col] = df[col].fillna(column_mean)

# REMOVE COMMENT BELOW TO ILLUSTRATE
#df

In [14]:
# Impute with mean using SimpleImputer
imputer = SimpleImputer(strategy='mean')
simple_imputed = pd.DataFrame(imputer.fit_transform(df_copy), columns=df_copy.columns)

# REMOVE COMMENT BELOW TO ILLUSTRATE
#simple_imputed

## Proximity-based methods
Use similar rows to estimate missing values.
>- **KNN**: Each sample’s missing values are imputed using the mean value from `n_neighbors` nearest neighbors found in the training set.
>- **Interpolation**: Missing values are estimated based on adjacent known data points, assuming a trend or pattern.

#### Example with KNN:

In [15]:
# Import KNNImputer from Imputer class
from sklearn.impute import KNNImputer

In [16]:
# Create sample data with nan value
data = {
    'income': [52000, 48000, 95000, 55000, 50000, 78000],
    'education': [16, 14, 20, 16, 15, 18],
    'age': [np.nan, 32, 45, 29, 35, 41]
}
df = pd.DataFrame(data)

df

,income,education,age
0,52000,16,NaN
1,48000,14,32.0
2,95000,20,45.0
3,55000,16,29.0
4,50000,15,35.0
5,78000,18,41.0


In [19]:
# Impute with KNNImputer
imputer = KNNImputer(n_neighbors=3)
knn_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# what value do you think will be used to fill age?
#knn_imputed

## Model-based prediction

Train a model on complete data, predict missing values.
>- **Linear regression**: Assumes linear relationships
>- **Random forest**: Captures non-linear patterns
>- **Iterative imputer**: Chains models across multiple columns with missing data

#### Example with Linear Regression:

In [117]:
# Import Linear Regression for 
from sklearn.linear_model import LinearRegression

In [118]:
# Create sample data with nan value
data = {
    'income': [52000, 48000, 95000, 55000, 50000, 78000],
    'education': [16, 14, 20, 16, 15, 18],
    'age': [np.nan, 32, 45, 29, 35, 41]
}
df = pd.DataFrame(data)

df

,income,education,age
0,52000,16,NaN
1,48000,14,32.0
2,95000,20,45.0
3,55000,16,29.0
4,50000,15,35.0
5,78000,18,41.0


In [119]:
missing_mask = df['age'].isna()
# Train on all rows but those missing value(s)
train = df[~missing_mask]
# Label is missing value
predict = df[missing_mask]

In [120]:
# Fit linear regression model
features = ['income', 'education']
model = LinearRegression()
model.fit(train[features], train['age'])

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [121]:
# Replace missing value with the model's predicted value
df.loc[missing_mask, 'age'] = model.predict(predict[features])

# REMOVE COMMENT BELOW TO ILLUSTRATE
#df

## The End